# Guardian of Truth v6p — фокусная проверка регламента (фрагмент политики)

**Гипотеза:** на длинном промпте (до 230 тыс. символов) модель теряет точечные правила политики; короткий запрос
с нужным фрагментом политики поймает процессные ошибки, которые P_base пропускает (зона P_base < 0.5).
Три варианта на Qwen3.6-35B-A3B-FP8 (как на платформе), на размеченных ходах dev (430) и τ² (903):
- `slice` — фрагмент политики + последняя реплика клиента + ответ агента (≈ 4 тыс. символов);
- `window` — фрагмент + последние сообщения диалога с результатами инструментов + ответ (≈ 10 тыс.);
- `focus` — полный промпт организаторов, но перед ответом агента ещё раз нужный фрагмент политики.

**Правило (записано до прогона):** вариант берём, если на τ² «H1 + вариант» даёт P@160 ≥ ×1.05 к H1 с нижней
границей бутстрэпа > ×1.0 и на dev не меньше ×1.0; или если на вызовах прирост AUC надёжно выше нуля и ≥ +2 ошибки
в топ-20%. Иначе не берём.

**Как запускать:** Colab, **A100 80GB или H100**, «Выполнить все» (на карте меньше 80 ГБ ноутбук сразу отключится).
**Оценка:** ≈ 30–40 мин ≈ **4–5 CU**. В конце среда отключится сама (сторожевой таймер — 1.5 ч).
**Результат:** вывод шага 6 и архив `MyDrive/guardian_of_truth/v6_policy/results_v6_policy.zip` — пришли Claude.

In [ ]:
# 1. Диск, папки, GPU (нужно 80 ГБ) + сторожевой таймер
import os, sys, json, time, subprocess, threading, shutil, re
from pathlib import Path
from google.colab import drive, runtime
drive.mount('/content/drive')
OUT = Path('/content/drive/MyDrive/guardian_of_truth/v6_policy'); (OUT / 'signals').mkdir(parents=True, exist_ok=True)
GENERATED = Path('/content/drive/MyDrive/guardian_of_truth/v5_tau2gen/examples_v5.parquet')
LOG = OUT / 'run_log.txt'

def note(message):
    line = time.strftime('%H:%M:%S ') + str(message)
    print(line, flush=True)
    with open(LOG, 'a', encoding='utf-8') as f:
        f.write(line + '\n')

def watchdog(seconds):
    time.sleep(seconds)
    note('сторожевой таймер: отключаю среду')
    try:
        drive.flush_and_unmount()
    finally:
        runtime.unassign()

threading.Thread(target=watchdog, args=(1.5 * 3600,), daemon=True).start()
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
                     capture_output=True, text=True).stdout.strip()
note('старт; GPU: ' + gpu)
if int(gpu.split(',')[-1]) < 75000:
    note('GPU меньше 80 ГБ — отключаю среду; перезапусти на A100 80GB или H100')
    drive.flush_and_unmount(); runtime.unassign()
    raise SystemExit('нужна видеокарта на 80 ГБ')
note('диалоги τ² на Диске: ' + ('есть' if GENERATED.exists() else 'НЕТ — будет только dev'))

In [ ]:
# 2. vLLM 0.19.1  (⏱ 3–6 мин)
import importlib.metadata as md
try:
    if md.version('vllm') != '0.19.1':
        raise md.PackageNotFoundError
except md.PackageNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'vllm==0.19.1'], check=False)
for package in ('vllm', 'transformers', 'torch'):
    note(f'{package} {md.version(package)}')

In [ ]:
# 3. Qwen3.6-35B-A3B-FP8 (без кэша кусков Hugging Face)  (⏱ 2–5 мин)
QWEN = Path('/content/models/qwen')
def download(repo, target):
    if (target / '.complete').exists():
        return True
    script = (f"from huggingface_hub import snapshot_download; snapshot_download({repo!r}, local_dir={str(target)!r}, "
              "allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja', 'merges.txt'])")
    result = subprocess.run([sys.executable, '-c', script], env={**os.environ, 'HF_HUB_DISABLE_XET': '1'},
                            capture_output=True, text=True)
    if result.returncode != 0:
        note(f'{repo} не скачалась:\n{result.stderr[-1500:]}'); return False
    (target / '.complete').write_text('ok'); return True
note('Qwen готова' if download('Qwen/Qwen3.6-35B-A3B-FP8', QWEN) else 'Qwen НЕ скачалась')

In [ ]:
# 4. Код, метки, наборы  (⏱ <1 мин)
import base64, io, tarfile
import pandas as pd
PAYLOAD_B64 = '...'
WORK = Path('/content/v6p'); WORK.mkdir(exist_ok=True)
INPUTS = {}
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD_B64)), mode='r:gz') as archive:
    archive.extractall(WORK)
subprocess.run([sys.executable, 'dev/make_pool.py', '--valid', 'valid.parquet', '--out', 'data'], cwd=WORK, check=True,
               capture_output=True, text=True)
INPUTS['dev'] = WORK / 'data' / 'dev.parquet'
if GENERATED.exists():
    wanted = set(json.loads((WORK / 'ids_tau2.json').read_text()))
    frame = pd.read_parquet(GENERATED)
    frame = frame[frame['id'].isin(wanted)][['id', 'prompt', 'response']].reset_index(drop=True)
    INPUTS['tau2'] = WORK / 'tau2_labeled.parquet'; frame.to_parquet(INPUTS['tau2'], index=False)
note('наборы: ' + ', '.join(f'{k} {len(pd.read_parquet(v))}' for k, v in INPUTS.items()))

In [ ]:
# 5. Сервер Qwen и сигналы slice / window / focus  (⏱ 5 мин + 15–20 мин)
import urllib.request

def stop_servers():
    subprocess.run('pkill -f "vllm.entrypoints.openai.api_server" || true', shell=True); time.sleep(15)
    subprocess.run('pkill -9 -f EngineCore || true', shell=True); time.sleep(3)

def start_server(timeout=1500):
    stop_servers()
    log = open('/content/vllm_qwen.log', 'w')
    process = subprocess.Popen([sys.executable, '-m', 'vllm.entrypoints.openai.api_server', '--model', str(QWEN),
                                '--served-model-name', 'm', '--port', '8000', '--max-model-len', '131072',
                                '--gpu-memory-utilization', '0.90', '--max-num-seqs', '128', '--enable-prefix-caching',
                                '--language-model-only'], stdout=log, stderr=subprocess.STDOUT)
    started = time.time()
    while time.time() - started < timeout and process.poll() is None:
        try:
            urllib.request.urlopen('http://127.0.0.1:8000/v1/models', timeout=5)
            note(f'сервер готов за {time.time() - started:.0f} с'); return True
        except Exception:
            time.sleep(5)
    shutil.copy('/content/vllm_qwen.log', OUT / 'vllm_qwen.log')
    lines = open('/content/vllm_qwen.log', encoding='utf-8', errors='replace').read().splitlines()
    first = next((i for i, l in enumerate(lines) if re.search(r'\w+Error\b|Traceback|out of memory', l) and 'Warning' not in l),
                 max(0, len(lines) - 30))
    note('сервер НЕ поднялся. Первая ошибка:\n' + '\n'.join(lines[first:first + 30])); return False

if start_server():
    for name, path in INPUTS.items():
        t = time.time()
        result = subprocess.run([sys.executable, 'dev/score_v6.py', '--input', str(path), '--name', name, '--model', 'qwen',
                                 '--base-url', 'http://127.0.0.1:8000/v1', '--out', str(OUT / 'signals'),
                                 '--signals', 'slice,window,focus'], cwd=WORK, capture_output=True, text=True)
        lines = [l for l in (result.stdout + result.stderr).splitlines() if l.strip() and '|' not in l]
        note(f'qwen/{name}: код {result.returncode}, {time.time() - t:.0f} с\n' + '\n'.join(lines[-6:]))
stop_servers()

In [ ]:
# 6. Сравнение и вердикт  (⏱ 1–2 мин)
result = subprocess.run([sys.executable, 'dev/eval_policy.py', '--labels', 'labels.json', '--signals', str(OUT / 'signals'),
                         '--out', str(OUT / 'summary.json')], cwd=WORK, capture_output=True, text=True)
print(result.stdout); print(result.stderr[-3000:])
(OUT / 'summary.txt').write_text(result.stdout + '\n' + result.stderr[-3000:], encoding='utf-8')

In [ ]:
# 7. Архив и отключение
import zipfile
with zipfile.ZipFile(OUT / 'results_v6_policy.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.rglob('*')):
        if path.is_file() and path.suffix in ('.json', '.jsonl', '.txt', '.parquet', '.log'):
            archive.write(path, path.relative_to(OUT).as_posix())
note('архив: ' + str(OUT / 'results_v6_policy.zip') + '; отключаю среду')
try:
    stop_servers(); drive.flush_and_unmount()
finally:
    runtime.unassign()